In [1]:
import pandas as pd

# Load the CSV file
df = pd.read_csv('ipl_bowl_features.csv')

# Get all column names
columns = df.columns.tolist()

# Find the index of 'venue' column
venue_index = columns.index('venue')

# Columns to rename (all columns after 'venue')
columns_to_rename = columns[venue_index + 1:]

# Create a mapping dictionary for renaming
rename_dict = {col: f"{col}_bowl" for col in columns_to_rename}

# Rename the columns
df = df.rename(columns=rename_dict)

# Save the modified dataframe back to CSV
df.to_csv('ipl_bowl_features_renamed.csv', index=False)

print("Column names after 'venue' have been updated with '_bowl' suffix.")

Column names after 'venue' have been updated with '_bowl' suffix.


In [2]:
import pandas as pd

# Load the CSV file
df = pd.read_csv('ipl_bat_features.csv')

# Get all column names
columns = df.columns.tolist()

# Find the index of 'venue' column
venue_index = columns.index('venue')

# Columns to rename (all columns after 'venue')
columns_to_rename = columns[venue_index + 1:]

# Create a mapping dictionary for renaming
rename_dict = {col: f"{col}_bat" for col in columns_to_rename}

# Rename the columns
df = df.rename(columns=rename_dict)

# Save the modified dataframe back to CSV
df.to_csv('ipl_bat_features_renamed.csv', index=False)

print("Column names after 'venue' have been updated with '_bat' suffix.")

Column names after 'venue' have been updated with '_bat' suffix.


In [3]:
import pandas as pd

# Load both datasets
bat_df = pd.read_csv('ipl_bat_features_renamed.csv')
bowl_df = pd.read_csv('ipl_bowl_features_renamed.csv')

# First, verify which columns actually exist in both datasets
print("Batting columns:", bat_df.columns.tolist())
print("\nBowling columns:", bowl_df.columns.tolist())

# Rename columns to common names
bat_df = bat_df.rename(columns={
    'batter_id': 'player_id',
    'batter': 'player',
    'season_bat': 'season'  # Rename season_bat to season
})

bowl_df = bowl_df.rename(columns={
    'bowler_id': 'player_id',
    'bowler': 'player',
    'season_bowl': 'season'  # Rename season_bowl to season
})

# Reorder columns to put season right after venue in both datasets
def reorder_columns(df):
    cols = df.columns.tolist()
    if 'season' in cols and 'venue' in cols:
        cols.remove('season')
        venue_index = cols.index('venue')
        cols.insert(venue_index + 1, 'season')
        return df[cols]
    return df

bat_df = reorder_columns(bat_df)
bowl_df = reorder_columns(bowl_df)

# Identify which common columns actually exist in both datasets
common_cols = ['date', 'match_id', 'player_id', 'player', 'venue', 'season', 'inns', 'bat_team', 'bowl_team']
existing_common_cols = [col for col in common_cols
                       if col in bat_df.columns and col in bowl_df.columns]

print("\nCommon columns that exist in both datasets:", existing_common_cols)

# Merge the datasets using only the existing common columns
merged_df = pd.merge(
    bat_df,
    bowl_df,
    on=existing_common_cols,
    how='outer',
    suffixes=('_bat', '_bowl')
)

# Handle team columns - create single columns by combining _bat and _bowl versions
for team_col in ['bat_team', 'bowl_team']:
    if f"{team_col}_bat" in merged_df.columns and f"{team_col}_bowl" in merged_df.columns:
        merged_df[team_col] = merged_df[f"{team_col}_bat"].fillna(merged_df[f"{team_col}_bowl"])
        merged_df.drop([f"{team_col}_bat", f"{team_col}_bowl"], axis=1, inplace=True)

# Reorder columns to have common info first
player_cols = [col for col in existing_common_cols if col in merged_df.columns]
other_cols = [col for col in merged_df.columns if col not in player_cols]
merged_df = merged_df[player_cols + other_cols]

# Save the merged dataset
merged_df.to_csv('ipl_player_features_merged_final.csv', index=False)

print("\nMerge completed successfully!")
print(f"Final columns ({len(merged_df.columns)}):", merged_df.columns.tolist())
print("\nFirst 2 rows:")
print(merged_df.head(2))

Batting columns: ['date', 'match_id', 'bat_team', 'bowl_team', 'inns', 'batter_id', 'batter', 'venue', 'career_avg_bat', 'career_rpm_bat', 'career_sr_bat', 'career_bpm_bat', 'career_bfpm_bat', 'inns_avg_bat', 'inns_rpm_bat', 'inns_sr_bat', 'inns_bpm_bat', 'inns_bfpm_bat', 'ven_avg_bat', 'ven_rpm_bat', 'ven_sr_bat', 'ven_bpm_bat', 'ven_bfpm_bat', 'season_bat', 'ssn_avg_bat', 'ssn_rpm_bat', 'ssn_sr_bat', 'ssn_bpm_bat', 'ssn_bfpm_bat', 'last5_avg_bat', 'last5_sr_bat', 'last5_rpm_bat', 'last5_bfpm_bat', 'last5_bpm_bat']

Bowling columns: ['date', 'match_id', 'bowl_team', 'bat_team', 'bowler_id', 'bowler', 'inns', 'venue', 'season_bowl', 'career_avg_bowl', 'career_eco_bowl', 'career_sr_bowl', 'career_wpm_bowl', 'career_opm_bowl', 'inns_avg_bowl', 'inns_eco_bowl', 'inns_sr_bowl', 'inns_wpm_bowl', 'inns_opm_bowl', 'venue_avg_bowl', 'venue_eco_bowl', 'venue_sr_bowl', 'venue_wpm_bowl', 'venue_opm_bowl', 'ssn_avg_bowl', 'ssn_eco_bowl', 'ssn_sr_bowl', 'ssn_wpm_bowl', 'ssn_opm_bowl', 'last5_avg_bo

In [4]:
import pandas as pd

# Load the merged dataset
merged_df = pd.read_csv('ipl_player_features_merged_final.csv')

# Identify columns to fill with 0 (all except these will remain unchanged)
columns_to_exclude = ['date', 'match_id', 'player_id', 'player', 'venue', 'bat_team', 'bowl_team']
columns_to_fill = [col for col in merged_df.columns if col not in columns_to_exclude]

# Fill specified columns with 0
merged_df[columns_to_fill] = merged_df[columns_to_fill].fillna(0)

# Save the cleaned dataset
merged_df.to_csv('ipl_player_features_merged_cleaned.csv', index=False)

print("Null values in performance metrics filled with 0")
print(f"Dataset saved to 'ipl_player_features_merged_cleaned.csv'")

Null values in performance metrics filled with 0
Dataset saved to 'ipl_player_features_merged_cleaned.csv'


In [5]:
import pandas as pd

# Load your merged dataset
df = pd.read_csv('ipl_player_features_merged_cleaned.csv')

# Print all column names
print("All column names in the dataset:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

# Print the number of columns
print(f"\nTotal columns: {len(df.columns)}")

# Optional: Print the first few rows to see sample data
print("\nSample data (first 2 rows):")
print(df.head(2))

All column names in the dataset:
1. date
2. match_id
3. player_id
4. player
5. venue
6. season
7. inns
8. bat_team
9. bowl_team
10. career_avg_bat
11. career_rpm_bat
12. career_sr_bat
13. career_bpm_bat
14. career_bfpm_bat
15. inns_avg_bat
16. inns_rpm_bat
17. inns_sr_bat
18. inns_bpm_bat
19. inns_bfpm_bat
20. ven_avg_bat
21. ven_rpm_bat
22. ven_sr_bat
23. ven_bpm_bat
24. ven_bfpm_bat
25. ssn_avg_bat
26. ssn_rpm_bat
27. ssn_sr_bat
28. ssn_bpm_bat
29. ssn_bfpm_bat
30. last5_avg_bat
31. last5_sr_bat
32. last5_rpm_bat
33. last5_bfpm_bat
34. last5_bpm_bat
35. career_avg_bowl
36. career_eco_bowl
37. career_sr_bowl
38. career_wpm_bowl
39. career_opm_bowl
40. inns_avg_bowl
41. inns_eco_bowl
42. inns_sr_bowl
43. inns_wpm_bowl
44. inns_opm_bowl
45. venue_avg_bowl
46. venue_eco_bowl
47. venue_sr_bowl
48. venue_wpm_bowl
49. venue_opm_bowl
50. ssn_avg_bowl
51. ssn_eco_bowl
52. ssn_sr_bowl
53. ssn_wpm_bowl
54. ssn_opm_bowl
55. last5_avg_bowl
56. last5_eco_bowl
57. last5_sr_bowl
58. last5_wpm_bowl
5

In [7]:
import pandas as pd
from tqdm import tqdm  # For progress tracking

def enhanced_fantasy_mapping():
    # Load datasets with dtype specification for memory efficiency
    dtype_spec = {
        'match_id': 'str',
        'player_id': 'str',
        'player': 'str'
    }

    # Load datasets with error handling
    try:
        merged_df = pd.read_csv('ipl_player_features_merged_cleaned.csv', dtype=dtype_spec)
        fantasy_df = pd.read_csv('player_match_fantasy_points_with_dream_team.csv', dtype=dtype_spec)
    except FileNotFoundError as e:
        print(f"Error loading files: {e}")
        return None

    # Data validation checks
    def validate_data(df, df_name):
        required_cols = {'match_id', 'player_id'}
        missing = required_cols - set(df.columns)
        if missing:
            print(f"Missing columns in {df_name}: {missing}")
            return False
        return True

    if not all([validate_data(merged_df, "merged data"),
               validate_data(fantasy_df, "fantasy data")]):
        return None

    # Create composite key for both datasets
    merged_df['composite_key'] = merged_df['match_id'] + '_' + merged_df['player_id']
    fantasy_df['composite_key'] = fantasy_df['match_id'] + '_' + fantasy_df['player_id']

    # Create lookup dictionary with progress bar
    print("\nCreating fantasy points lookup dictionary...")
    fantasy_lookup = {}
    for _, row in tqdm(fantasy_df.iterrows(), total=len(fantasy_df)):
        fantasy_lookup[row['composite_key']] = row.get('final_dt', None)

    # Map values with comprehensive checks
    print("\nMapping fantasy points to merged data...")
    merged_df['final_dt'] = None

    missing_count = 0
    for idx, row in tqdm(merged_df.iterrows(), total=len(merged_df)):
        key = row['composite_key']
        if key in fantasy_lookup:
            merged_df.at[idx, 'final_dt'] = fantasy_lookup[key]
        else:
            missing_count += 1

    # Post-mapping validation
    total_rows = len(merged_df)
    matched_rows = total_rows - missing_count
    match_percentage = (matched_rows / total_rows) * 100

    print(f"\nResults:")
    print(f"Total rows processed: {total_rows}")
    print(f"Successfully matched: {matched_rows} ({match_percentage:.2f}%)")
    print(f"Missing matches: {missing_count}")

    if missing_count > 0:
        # Create DataFrame of unmatched entries for debugging
        unmatched_df = merged_df[merged_df['final_dt'].isna()][['match_id', 'player_id', 'player']]
        print("\nSample of unmatched entries:")
        print(unmatched_df.head())

        # Save unmatched entries for review
        unmatched_df.to_csv('unmatched_entries.csv', index=False)
        print("Saved unmatched entries to 'unmatched_entries.csv'")

    # Save the enhanced dataset
    output_path = 'ipl_players_features_merged_enhanced.csv'
    merged_df.drop(columns=['composite_key'], inplace=True)
    merged_df.to_csv(output_path, index=False)

    print(f"\nEnhanced dataset saved to {output_path}")
    return merged_df

# Execute the function
final_df = enhanced_fantasy_mapping()
if final_df is not None:
    print("\nFirst 5 rows of final dataset:")
    print(final_df[['match_id', 'player_id', 'player', 'final_dt']].head())


Creating fantasy points lookup dictionary...


100%|██████████| 24701/24701 [00:01<00:00, 17822.39it/s]



Mapping fantasy points to merged data...


100%|██████████| 48777/48777 [00:03<00:00, 12864.22it/s]



Results:
Total rows processed: 48777
Successfully matched: 48777 (100.00%)
Missing matches: 0

Enhanced dataset saved to ipl_players_features_merged_enhanced.csv

First 5 rows of final dataset:
  match_id player_id          player final_dt
0   336001     44936  AB de Villiers        1
1   336004     44936  AB de Villiers        1
2   336009     44936  AB de Villiers        0
3   336015     44936  AB de Villiers        1
4   336017     44936  AB de Villiers        0


In [7]:
!pip uninstall cuml -y
!pip uninstall pycaret -y
!pip install pycaret

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.7/169.7 kB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of category-encoders to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.1/486.1 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.8/21.8 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.2/302.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 87.0 MB/s eta 0:00:00
   ━━━━

In [8]:
import pandas as pd
from pycaret.classification import *
from sklearn.preprocessing import LabelEncoder
import joblib

# Load the enhanced dataset
df = pd.read_csv('ipl_players_features_merged_enhanced.csv')

# Drop rows with missing target
df = df.dropna(subset=['final_dt'])

# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
categorical_cols = [col for col in categorical_cols if col not in ['match_id', 'player_id', 'player', 'final_dt', 'date']]

# Columns to ignore/drop
cols_to_ignore = ['date', 'match_id', 'player']
existing_cols_to_ignore = [col for col in cols_to_ignore if col in df.columns]

# Label Encoding for categorical features
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

# Manual Train-Validation Split (based on season)
df_train = df[df['season'] < 2023].copy()
df_val = df[df['season'] >= 2023].copy()

# Setup PyCaret for training data only
clf_setup = setup(
    data=df_train,
    target='final_dt',
    session_id=42,
    categorical_features=categorical_cols,
    normalize=True,
    transformation=True,
    ignore_features=existing_cols_to_ignore,
    verbose=False
)

# Train and tune models
best_models = compare_models(sort='Accuracy', n_select=3, fold=3, exclude=['dummy'])

tuned_models = []
for model in best_models:
    tuned = tune_model(model, optimize='Accuracy', n_iter=5, choose_better=True)
    tuned_models.append(tuned)

# Finalize the best model from training
final_model = automl(optimize='Accuracy')

# Save model and encoders
save_model(final_model, 'ipl_fantasy_classifier')
joblib.dump(label_encoders, 'label_encoders.pkl')

# Evaluate on validation set
predictions = predict_model(final_model, data=df_val)

# Inverse transform categorical predictions
for col in categorical_cols:
    if col in predictions.columns:
        predictions[col] = label_encoders[col].inverse_transform(predictions[col])

# Save predictions
output_cols = ['player_id', 'final_dt', 'prediction_label'] + [col for col in categorical_cols if col in predictions.columns]
predictions[output_cols].to_csv('classification_predictions.csv', index=False)

print("\nTraining completed! Model and encoders saved.")
print(f"Final model type: {type(final_model).__name__}")


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
gbc,Gradient Boosting Classifier,0.6782,0.5906,0.9929,0.6794,0.8068,0.0175,0.0569,14.4400
lightgbm,Light Gradient Boosting Machine,0.6779,0.5898,0.9684,0.6854,0.8027,0.0502,0.0889,5.9633
ada,Ada Boost Classifier,0.6765,0.5877,0.9911,0.6786,0.8056,0.0130,0.0416,5.2967
ridge,Ridge Classifier,0.6764,0.5647,0.9962,0.6774,0.8064,0.0053,0.0262,2.3500
lr,Logistic Regression,0.6763,0.5640,0.9955,0.6775,0.8062,0.0059,0.0262,6.1333
lda,Linear Discriminant Analysis,0.6761,0.5647,0.9939,0.6777,0.8059,0.0074,0.0290,2.3067
svm,SVM - Linear Kernel,0.6753,0.5092,0.9968,0.6764,0.8060,-0.0003,0.0003,3.9900
xgboost,Extreme Gradient Boosting,0.6598,0.5716,0.9065,0.6889,0.7829,0.0611,0.0755,3.8967
rf,Random Forest Classifier,0.6550,0.5505,0.8891,0.6901,0.7771,0.0646,0.0761,9.6833
et,Extra Trees Classifier,0.6414,0.5297,0.8583,0.6885,0.7641,0.0533,0.0594,8.1467


Processing:   0%|          | 0/63 [00:00<?, ?it/s]

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.6762,0.6003,0.9919,0.6783,0.8057,0.0104,0.0355
1,0.6789,0.5924,0.9955,0.6793,0.8075,0.0165,0.0628
2,0.6779,0.5872,0.9939,0.6790,0.8068,0.0145,0.0521
3,0.6755,0.5841,0.9929,0.6776,0.8055,0.0061,0.0231
4,0.6779,0.5727,0.9939,0.6790,0.8068,0.0145,0.0521
5,0.6776,0.5927,0.9934,0.6787,0.8065,0.0152,0.0528
6,0.6775,0.5941,0.9904,0.6795,0.8060,0.0182,0.0542
7,0.6744,0.5962,0.9894,0.6776,0.8043,0.0070,0.0222
8,0.6788,0.5717,0.9950,0.6793,0.8074,0.0173,0.0631


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

Fitting 10 folds for each of 5 candidates, totalling 50 fits


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.6766,0.5797,1.0000,0.6766,0.8071,0.0000,0.0000
1,0.6766,0.5933,1.0000,0.6766,0.8071,0.0000,0.0000
2,0.6766,0.5788,1.0000,0.6766,0.8071,0.0000,0.0000
3,0.6766,0.5810,1.0000,0.6766,0.8071,0.0000,0.0000
4,0.6766,0.5675,1.0000,0.6766,0.8071,0.0000,0.0000
5,0.6762,0.5811,1.0000,0.6762,0.8068,0.0000,0.0000
6,0.6765,0.5819,1.0000,0.6765,0.8070,0.0000,0.0000
7,0.6765,0.5907,1.0000,0.6765,0.8070,0.0000,0.0000
8,0.6765,0.5851,1.0000,0.6765,0.8070,0.0000,0.0000


... (5 outputs left out)

Fitting 10 folds for each of 5 candidates, totalling 50 fits


Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).
Transformation Pipeline and Model Successfully Saved


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Light Gradient Boosting Machine,0.6280,0.5605,0.9559,0.6352,0.7632,0.0387,0.0674



Training completed! Model and encoders saved.
Final model type: LGBMClassifier
